In [2]:
import os
import cv2
import numpy as np
import torch
from network_scunet import SCUNet
import tqdm as notebook_tqdm

In [27]:
def denoise_clock(
    image_path: str,
    model_name: str,
    output_path: str,
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
):
    model_dir = os.path.join(os.path.dirname(os.getcwd()), "models")
    model_path = os.path.join(model_dir, model_name)
    
    # determine channel config
    is_gray = "gray" in model_name
    in_nc = 1 if is_gray else 3
    
    if "real" in model_name:
        config = [4,4,4,4,4,4,4]
    else:
        config = [2,2,2,2,2,2,2]
    
    model = SCUNet(in_nc=in_nc, config=config, dim=64)
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True), strict=True)
    model.eval().to(device)
    
    # load and preprocess image
    if is_gray:
        img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        img = np.expand_dims(img, axis=2)
    else:
        img = cv2.imread(image_path, cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
    # convert to tensor 
    img_tensor = torch.from_numpy(np.ascontiguousarray(img)).permute(2, 0, 1).float() / 255.0
    img_tensor = img_tensor.unsqueeze(0).to(device)
    
    # Run inference
    with torch.no_grad():
        output_tensor = model(img_tensor)

    # Post-process tensor back to image
    output_img = output_tensor.squeeze().cpu().numpy()
    output_img = np.clip(output_img * 255.0, 0, 255).astype(np.uint8)

    if is_gray:
        cv2.imwrite(output_path, output_img)
    else:
        output_img = output_img.transpose(1, 2, 0)
        output_img = cv2.cvtColor(output_img, cv2.COLOR_RGB2BGR)
        cv2.imwrite(output_path, output_img)

    print(f"Processed with '{model_name}' -> Saved to {output_path}")
    
denoise_clock("./../datasets/denoise_test.jpg", model_name="scunet_color_real_psnr.pth", output_path="out_psnr.jpg")
denoise_clock("./../datasets/denoise_test.jpg", model_name="scunet_color_real_gan.pth", output_path="out_gan.jpg")

Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Blo